# 05 · Plug in your own context sensing

ContextXR takes context as given (paper §5.2). A real system would *sense* it, for example with an activity recognizer or a vision-language model, and the sensed labels are free-form. `ContextNormalizer` maps each free-form label to the closest task or environment label in the benchmark vocabulary, using all-MiniLM-L6-v2 sentence embeddings. Any method can then use the mapped label. Needs the `[embed]` extra.

In [ ]:
from contextxr import load_minexrpp
from contextxr.normalize import ContextNormalizer

ds = load_minexrpp()
norm = ContextNormalizer(ds)
for query in ["making pasta", "chilling on the sofa", "answering emails at my desk"]:
    m = norm.task(query)
    print(f"task  {query!r:32} -> {m.label:14} ({m.similarity:.2f}); next: {[a for a, _ in m.alternatives[:2]]}")
for query in ["home kitchen", "cafe", "open-plan office"]:
    m = norm.environment(query)
    print(f"env   {query!r:32} -> {m.label:14} ({m.similarity:.2f})")

## Evaluate with sensed contexts

Simulate a sensing module that describes some tasks and every environment in its own words. Normalize the labels, swap them into the dataset, and evaluate the same method on the true contexts and on the sensed ones.

In [ ]:
sensed_task = {"cooking": "making pasta", "relaxing": "chilling on the sofa", "working": "answering emails at my desk",
               "studying": "studying for an exam", "readingbook": "reading a novel"}
sensed_env = {"kitchen": "home kitchen", "livingroom": "living room couch", "office": "open-plan office", "coffeeshop": "cafe"}

contexts = {}
for sid, s in ds.scenarios.items():
    task = sensed_task.get(s.context.task.lower(), s.context.task)
    env = sensed_env[s.context.environment.lower()]
    contexts[sid] = norm.context(s.context.user, task, env)

changed = sum(contexts[sid].task != s.context.task or contexts[sid].environment != s.context.environment
              for sid, s in ds.scenarios.items())
print(f"{changed} of {len(ds)} contexts differ from the ground-truth labels after normalization")
ds_sensed = ds.with_contexts(contexts)

In [ ]:
from IPython.display import Markdown
from contextxr import evaluate
from contextxr.methods import RelationalRetrieval
from contextxr.runner import format_table

true_ctx = evaluate(RelationalRetrieval(), "initial", ds)
sensed = evaluate(RelationalRetrieval(), "initial", ds_sensed)
true_ctx.run["method"]["name"] = "relational, true context"
sensed.run["method"]["name"] = "relational, sensed context"
Markdown(format_table([true_ctx, sensed]))

| method | n | manual | cost | reduction | Recall@10 | F1@10 | Hit@10 |
|---|---:|---:|---:|---:|---:|---:|---:|
| relational, true context | 109 | 21.20 | 18.39 | -13.3% | 0.259 | 0.193 | 0.798 |
| relational, sensed context | 109 | 21.20 | 18.18 | -14.3% | 0.275 | 0.198 | 0.798 |

Differences between the two rows reflect what is lost, or gained, by mapping sensed labels onto the benchmark vocabulary. For example, `Coffeeshop` and `CoffeeShop` collapse into one label.